In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# folder dbscan/ of the project: the c++ programs live there and the csv files go to dbscan/data.
# It is searched from the working directory of the kernel, so the notebook works when it is run
# from computoparalelo2026b/, from dbscan/ or from the parent folder.
here = Path.cwd().resolve()
DBSCAN_DIR = next((p for p in (here, here/"dbscan", here/"computoparalelo2026b"/"dbscan", here.parent/"dbscan")
                   if (p/"dbscan_p1.cpp").exists()), None)
assert DBSCAN_DIR is not None, f"dbscan/ folder not found from {here}"
DATA_DIR = DBSCAN_DIR/"data"
DATA_DIR.mkdir(exist_ok=True)

# Indicate the number of points
n_points = 4000

# randomly generating data points and noise
points, y_true = make_blobs(n_samples=n_points,
                            centers=4,
                            cluster_std=0.06,
                            random_state=11,
                            center_box=(0, 1.0))

# only positive points and with three decimals
points = np.round(np.abs(points[:, ::-1]), 3)

# storing points into a csv file
np.savetxt(DATA_DIR/(str(n_points)+"_data.csv"), points, delimiter=",",  fmt="%.3f")

# clustering and detecting noise with dbscan
clusters = DBSCAN(eps=0.03, min_samples=10).fit_predict(points)

# plotting noise
plt.figure()
plt.title("Detecting noise with DBSCAN")
plt.scatter(points[:, 0], points[:, 1], c=np.where(clusters==-1, 0, 1), s=50)
plt.xticks([])
plt.yticks([])
plt.box(False)
plt.show()


In [ ]:
# running my c++ DBSCAN (serial, P1 and P2) on the csv file of the previous cell.
# Each program writes its own result csv and prints only its time in seconds.
# The version in PLOT is also copied to N_results.csv, the file the next cell plots.
import os
import shutil
import subprocess

PLOT = "p2"
threads = os.cpu_count()
runs = {"serial": [], "p1": [threads], "p2": [threads, 16]}   # extra arguments: threads (and k)

subprocess.run(["make"], cwd=DBSCAN_DIR, check=True, stdout=subprocess.DEVNULL)   # builds only if needed
data_csv = DATA_DIR/(str(n_points)+"_data.csv")
for version, extra in runs.items():
    out_csv = DATA_DIR/f"{n_points}_results_{version}.csv"
    command = [DBSCAN_DIR/f"dbscan_{version}", data_csv, out_csv] + [str(a) for a in extra]
    seconds = float(subprocess.run(command, capture_output=True, text=True, check=True).stdout)
    print(f"{version:6s} {str(extra):8s} {seconds:.4f} s  -> {out_csv.name}")
shutil.copy(DATA_DIR/f"{n_points}_results_{PLOT}.csv", DATA_DIR/(str(n_points)+"_results.csv"))


In [ ]:

# loading results from c++ program into a numpy array
result = np.loadtxt(DATA_DIR/(str(n_points)+"_results.csv"), delimiter=",")

# plotting noise
plt.figure()
plt.title("Detecting noise with my DBSCAN")
plt.scatter(result.T[0], result.T[1], c=result.T[2], s=50)
plt.xticks([])
plt.yticks([])
plt.box(False)
plt.show()

In [ ]:
# checking every result csv of my c++ DBSCAN against sklearn, label by label (same rule as dbscan/check.py):
#   0 noise <-> sklearn noise, 1 core of first order <-> sklearn core, 2 core of second order <-> sklearn border
db = DBSCAN(eps=0.03, min_samples=10).fit(points)
expected = np.zeros(n_points, dtype=int)
expected[db.labels_ != -1] = 2
expected[db.core_sample_indices_] = 1

for csv in sorted(DATA_DIR.glob(f"{n_points}_results*.csv")):
    result = np.loadtxt(csv, delimiter=",")
    # the output must keep the points in the same order as the input
    assert len(points) == len(result) and np.allclose(points, result[:, :2]), f"{csv.name}: points differ"
    mismatches = int(np.sum(expected != result[:, 2]))
    print(f"{csv.name:24s} [noise, core1, core2] = {np.bincount(result[:, 2].astype(int), minlength=3).tolist()}  "
          f"mismatches vs sklearn: {mismatches}  {'OK' if mismatches == 0 else 'MISMATCH'}")